# Bureau d'Analyse Terrestre — Klaxo-3

Notebook principal : Phases 1 à 6 du projet OVNI.

Fichier source : `releves_klaxo3.csv` (sans en-têtes, 11 colonnes).

Exécuter toutes les cellules dans l'ordre (Run All).

## Setup — imports et constantes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

FICHIER_SOURCE = "releves_klaxo3.csv"
FICHIER_NETTOYE = "releves_klaxo3_nettoye.csv"

COLONNES = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

CANULAR_PATTERNS = [
    r"(?i)possible hoax",
    r"(?i)student report",
    r"(?i)editorial comment",
    r"(?i)does not look like a ufo",
]

---
# Phase 1 — Ouvrir la caisse

Objectif : charger la transmission et rendre **3 nombres cohérents**.
- Lignes dans le fichier
- Lignes chargées
- Lignes traitées à part (mal formées)

In [ ]:
lignes_a_part = []

def bad_handler(bad_line):
    lignes_a_part.append(bad_line)
    return None

df_brut = pd.read_csv(
    FICHIER_SOURCE,
    header=None,
    names=COLONNES,
    engine="python",
    on_bad_lines=bad_handler,
)

nb_lignes_fichier = sum(1 for _ in open(FICHIER_SOURCE, encoding="utf-8", errors="replace"))
nb_chargees = len(df_brut)
nb_a_part = len(lignes_a_part)

print(f"Lignes dans le fichier      : {nb_lignes_fichier:,}")
print(f"Lignes chargées             : {nb_chargees:,}")
print(f"Lignes traitées à part      : {nb_a_part:,}")
print(f"Vérification (chargées+à part): {nb_chargees + nb_a_part:,}")

In [ ]:
if lignes_a_part:
    ex = lignes_a_part[0]
    print("Exemple de ligne traitée à part :")
    print(f"  Champs détectés : {len(ex)} (attendu : 11)")
    print(f"  Contenu : {ex}")
    print("  Problème : virgules en trop / champs vides décalés.")

### Premier regard (exploration)

In [ ]:
print(f"Dimensions : {df_brut.shape[0]:,} lignes x {df_brut.shape[1]} colonnes")
display(df_brut.head(10))

In [ ]:
missing = pd.DataFrame({
    "colonne": df_brut.columns,
    "nb_nan": df_brut.isna().sum().values,
    "pct_nan": (df_brut.isna().sum().values / len(df_brut) * 100).round(2),
}).sort_values("nb_nan", ascending=False)
display(missing[missing["nb_nan"] > 0])

---
# Phase 2 — Conversion de types

**Aucune ligne supprimée** — on convertit chaque champ au bon type et on compte les anomalies.

In [ ]:
df = df_brut.copy()

df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")

for col in ["city", "state", "country", "shape", "duration_hours_min", "comments"]:
    df[col] = df[col].astype("string")

print(f"Lignes conservées : {len(df):,} (aucune suppression)")
df[["duration_seconds", "latitude", "longitude"]].describe()

In [ ]:
anomalies_rows = []

def add_anomaly(nom, mask, source, exemple_fn):
    n = int(mask.sum())
    ex = exemple_fn() if n > 0 else "—"
    anomalies_rows.append({
        "anomalie": nom, "nombre": n,
        "pct": round(n / len(df) * 100, 2), "source": source, "exemple": ex,
    })

ville_vide = df["city"].isna() | (df["city"].str.strip() == "")
add_anomaly("Ville vide", ville_vide, "Témoin / transmission",
            lambda: df.loc[ville_vide, ["city", "state"]].head(1).to_dict())

coords_zero = (df["latitude"] == 0) & (df["longitude"] == 0)
add_anomaly("Coordonnées (0, 0)", coords_zero, "Capteur / géolocalisation",
            lambda: df.loc[coords_zero, ["city", "latitude", "longitude"]].head(1).to_dict())

duree_zero = df["duration_seconds"] == 0
add_anomaly("Durée = 0 seconde", duree_zero, "Service de transmission",
            lambda: df.loc[duree_zero, ["duration_seconds", "duration_hours_min"]].head(1).to_dict())

datetime_invalid = df["datetime"].isna()
add_anomaly("Date d'observation illisible", datetime_invalid, "Témoin / transmission",
            lambda: str(df_brut.loc[datetime_invalid].head(1)["datetime"].values[0]))

pays_vide = df["country"].isna() | (df["country"].str.strip() == "")
add_anomaly("Pays vide", pays_vide, "Témoin",
            lambda: str(df.loc[pays_vide, "city"].head(1).values[0]))

anomalies = pd.DataFrame(anomalies_rows)
display(anomalies)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=anomalies, x="nombre", y="anomalie", ax=ax, color="steelblue")
ax.set_title("Anomalies détectées (Phase 2 — sans suppression)")
ax.set_xlabel("Nombre de lignes")
plt.tight_layout()
plt.show()

---
# Phase 3 — Règle de canulars

**Règle :** un relevé est un canular si son commentaire contient une note NUFORC mentionnant un hoax, un rapport étudiant, un commentaire éditorial, ou indiquant que ce n'est pas un UFO.

**Limite :** la règle ne voit que ce que le modérateur a annoté — un canular sans note passe inaperçu.

In [ ]:
comments = df["comments"].astype(str)
is_canular = pd.Series(False, index=df.index)
for pattern in CANULAR_PATTERNS:
    is_canular |= comments.str.contains(pattern, regex=True, na=False)

n_canular = int(is_canular.sum())
pct_canular = n_canular / len(df) * 100
print(f"Canulars détectés : {n_canular:,} ({pct_canular:.2f} %)")
display(df.loc[is_canular, ["datetime", "city", "shape", "comments"]].head(5))

---
# Phase 4 — Modèle (avec comments, avant correction fuite)

Random Forest sur 80 % train / 20 % test (jamais vus).

In [ ]:
def preparer_features(data, avec_commentaires=False):
    X = data[["city", "state", "country", "shape", "duration_seconds", "latitude", "longitude"]].copy()
    X["duration_seconds"] = X["duration_seconds"].fillna(0)
    X["latitude"] = X["latitude"].fillna(0)
    X["longitude"] = X["longitude"].fillna(0)
    for col in ["city", "state", "country", "shape"]:
        X[col] = X[col].fillna("").astype(str)
    if avec_commentaires:
        X["comments"] = data["comments"].fillna("").astype(str)
    return X

def construire_modele(avec_commentaires=False):
    cat_cols = ["city", "state", "country", "shape"]
    num_cols = ["duration_seconds", "latitude", "longitude"]
    transformers = [
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="")),
            ("oh", OneHotEncoder(handle_unknown="ignore", max_categories=40)),
        ]), cat_cols),
        ("num", "passthrough", num_cols),
    ]
    if avec_commentaires:
        transformers.append(("txt", TfidfVectorizer(max_features=300, ngram_range=(1, 2)), "comments"))
    return Pipeline([
        ("prep", ColumnTransformer(transformers)),
        ("clf", RandomForestClassifier(n_estimators=80, max_depth=14, random_state=42,
                                       n_jobs=-1, class_weight="balanced")),
    ])

def entrainer_evaluer(data, y, avec_commentaires=False):
    X = preparer_features(data, avec_commentaires)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y.astype(int), test_size=0.2, random_state=42, stratify=y
    )
    modele = construire_modele(avec_commentaires)
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    return {
        "recall": recall_score(y_test, pred, zero_division=0),
        "precision": precision_score(y_test, pred, zero_division=0),
        "accuracy": accuracy_score(y_test, pred),
        "y_test": y_test, "pred": pred,
    }

In [ ]:
metriques_avant = entrainer_evaluer(df, is_canular, avec_commentaires=True)
print(f"Rappel    : {metriques_avant['recall']*100:.1f} %")
print(f"Précision : {metriques_avant['precision']*100:.1f} %")
print(f"Accuracy  : {metriques_avant['accuracy']*100:.1f} %")

---
# Phase 5 — Fuite de données

| Colonne | Qui écrit | Quand | Savait si canular ? |
|---|---|---|---|
| datetime | Témoin | Soir de l'observation | Non |
| city/state/country | Témoin | Observation | Non |
| shape | Témoin | Observation | Non |
| duration_seconds | Service transmission | Traitement | Non |
| latitude/longitude | Capteur Klaxo-3 | Observation | Non |
| comments | Témoin + employé Bureau | Mixte | **Oui** |
| date_posted | Employé Bureau | Semaines après | **Oui** |

On retire `comments` et `date_posted` du modèle.

In [ ]:
metriques_apres = entrainer_evaluer(df, is_canular, avec_commentaires=False)

comparaison = pd.DataFrame({
    "Métrique": ["Rappel", "Précision"],
    "Avant (avec comments)": [
        f"{metriques_avant['recall']*100:.1f} %",
        f"{metriques_avant['precision']*100:.1f} %",
    ],
    "Après (sans fuite)": [
        f"{metriques_apres['recall']*100:.1f} %",
        f"{metriques_apres['precision']*100:.1f} %",
    ],
})
display(comparaison)

print("\nExplication : le modèle « avant » lisait les notes du modérateur dans comments.")
print("L'employé savait déjà si c'était un canular — c'est de la triche.")
print("Sans comments, le modèle est honnête mais attrape moins de canulars.")

---
# Phase 6 — Baseline stagiaire

Le stagiaire répond toujours « pas un canular ». Son accuracy est élevée parce qu'il ignore tous les canulars — d'où l'importance du **rappel** et de la **précision**.

In [ ]:
y_test = metriques_apres["y_test"]
pred_stagiaire = np.zeros_like(y_test)

acc_stagiaire = accuracy_score(y_test, pred_stagiaire)
acc_modele = metriques_apres["accuracy"]

print(f"Stagiaire (toujours pas canular) : {acc_stagiaire*100:.1f} % accuracy")
print(f"Notre modèle (sans fuite)        : {acc_modele*100:.1f} % accuracy")
print(f"Rappel modèle                    : {metriques_apres['recall']*100:.1f} %")
print(f"Précision modèle                 : {metriques_apres['precision']*100:.1f} %")

---
# Annexe — Nettoyage pour cartographie (hors sujet noté)

Filtre les relevés exploitables pour une carte : lieu valide + durée > 0.

In [ ]:
lieu_invalide = (
    df["city"].isna() | (df["city"].str.strip() == "")
    | ((df["latitude"] == 0) & (df["longitude"] == 0))
)
temps_invalide = df["duration_seconds"] == 0
ok = ~lieu_invalide & ~temps_invalide

print(f"Lignes exploitables : {ok.sum():,} / {len(df):,}")
print(f"Lignes à supprimer  : {(~ok).sum():,}")

In [ ]:
df_clean = df[ok].copy()
df_clean.to_csv(FICHIER_NETTOYE, index=False)
print(f"Export : {FICHIER_NETTOYE} ({len(df_clean):,} lignes)")

### Synthèse visuelle (exploration qualité)

In [ ]:
categories = pd.Series({
    "OK (exploitable)": ok.sum(),
    "Lieu invalide seulement": (lieu_invalide & ~temps_invalide).sum(),
    "Temps invalide seulement": (~lieu_invalide & temps_invalide).sum(),
    "Les deux invalides": (lieu_invalide & temps_invalide).sum(),
})

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=categories.index, y=categories.values, ax=ax, palette="Set2")
ax.set_title("Répartition qualité des relevés")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()